In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import copy

# 1. Define a basic Lightweight Neural Network for Intrusion Detection
class SimpleMLP(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(SimpleMLP, self).__init__()
        # A simple lightweight architecture suitable for IoT intrusion detection
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, num_classes)
        
    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        return out

# 2. Simulated Client Training Function
def client_update(client_model, optimizer, train_loader, epochs=1):
    """Trains the model locally on the client's data."""
    client_model.train()
    criterion = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        for data, target in train_loader:
            optimizer.zero_grad()
            output = client_model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
    return client_model.state_dict()

# 3. Federated Averaging (FedAvg) Server Function
def server_aggregate(global_model, client_weights):
    """Aggregates client model weights by calculating their mean."""
    global_dict = global_model.state_dict()
    for k in global_dict.keys():
        # Averages the weights for each layer across all clients
        global_dict[k] = torch.stack([client_weights[i][k].float() for i in range(len(client_weights))], 0).mean(0)
    global_model.load_state_dict(global_dict)
    return global_model

# --- SIMULATION SETUP ---
num_clients = 3
num_rounds = 5
input_size = 115 # Example: N-BaIoT dataset has 115 features
hidden_size = 32
num_classes = 2  # Binary classification (Benign vs Malicious)

# Initialize Global Model
global_model = SimpleMLP(input_size, hidden_size, num_classes)

# Create Dummy Data for 3 Clients 
# (In practice, you will replace this with your actual N-BaIoT data for each IoT device)
client_loaders = []
for _ in range(num_clients):
    features = torch.randn(100, input_size)
    labels = torch.randint(0, num_classes, (100,))
    dataset = TensorDataset(features, labels)
    client_loaders.append(DataLoader(dataset, batch_size=16, shuffle=True))

# --- FEDERATED LEARNING LOOP ---
print("Starting Federated Learning Simulation...")
for round_idx in range(num_rounds):
    client_weights = []
    
    # Train on each client simulating local IoT device training
    for i in range(num_clients):
        # Give client a copy of the current global model
        client_model = copy.deepcopy(global_model)
        optimizer = optim.SGD(client_model.parameters(), lr=0.01)
        
        # Local training
        weights = client_update(client_model, optimizer, client_loaders[i], epochs=2)
        client_weights.append(weights)
        
    # Aggregate weights on the server to form the new global model
    global_model = server_aggregate(global_model, client_weights)
    print(f"Round {round_idx + 1} completed.")

print("Federated Learning Training Finished!")